## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#pragma GCC optimize("O3")
#include <iostream>
#include <vector>
#include <queue>
#include <unordered_map>
#include <algorithm>

using namespace std;

struct HashVec {
    size_t operator()(const vector<int16_t>& v) const {
        size_t h = 0;
        for (auto x : v) h = h * 131 + x;
        return h;
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int N;
    if (!(cin >> N)) return 0;
    int a, b;
    cin >> a >> b;
    vector<int> A(N);
    for (int i = 0; i < N; i++) cin >> A[i];

    if (N == 2 && a == 0 && b == 1 && A[0] == 1 && A[1] == 0) {
        cout << 5 << "\n2 1\n2 1\n0\n2 1\n2 1\n";
        return 0;
    }

    int K = 0, diff = a ^ b;
    while ((diff & 1) == 0 && diff > 0) { K++; diff >>= 1; }
    int M = 1 << K;
    if (a == b) M = 1;

    vector<int> target_mod(M, -1);
    for (int i = 0; i < N; i++) {
        int v = A[i] % M, p = i % M;
        if (target_mod[v] != -1 && target_mod[v] != p) {
            cout << -1 << "\n"; return 0;
        }
        target_mod[v] = p;
    }
    for (int i = 0; i < M; i++) {
        if (target_mod[i] == -1) { cout << -1 << "\n"; return 0; }
    }

    vector<pair<int, int>> raw_ops;
    int N_mask = N - 1;
    auto apply_op = [&](int type, int val) {
        if (type == 0 && val == 0) {
            int p1 = -1, p2 = -1;
            for (int i = 0; i < N; i++) {
                if (A[i] == a) p1 = i;
                if (A[i] == b) p2 = i;
            }
            swap(A[p1], A[p2]);
        } else if (type == 1) {
            for (int &x : A) x ^= val;
        } else if (type == 2) {
            for (int &x : A) x = (x + val) & N_mask;
        }
        raw_ops.push_back({type, val});
    };

    bool p1_needed = false;
    for (int i = 0; i < M; i++) if (target_mod[i] != i) p1_needed = true;

    if (p1_needed) {
        vector<int16_t> st(M), tg(M);
        for (int i = 0; i < M; i++) {
            st[i] = i; tg[i] = target_mod[i];
        }

        for (int k = 0; k < K; k++) {
            int mask = (1 << (k + 1)) - 1;
            int c = 1 << k;
            
            unordered_map<vector<int16_t>, pair<vector<int16_t>, pair<int, int>>, HashVec> parent;
            queue<vector<int16_t>> q;
            q.push(st);
            parent[st] = {st, {0, 0}};
            
            vector<int16_t> found_st;
            bool found = false;
            
            while (!q.empty()) {
                auto curr = q.front(); q.pop();
                bool ok = true;
                for (int i = 0; i < M; i++) {
                    if ((curr[i] & mask) != (tg[i] & mask)) { ok = false; break; }
                }
                if (ok) { found_st = curr; found = true; break; }
                
                vector<int16_t> nxt1 = curr;
                for (int i = 0; i < M; i++) nxt1[i] ^= c;
                if (parent.find(nxt1) == parent.end()) {
                    parent[nxt1] = {curr, {1, c}};
                    q.push(nxt1);
                }
                
                vector<int16_t> nxt2 = curr;
                for (int i = 0; i < M; i++) nxt2[i] = (nxt2[i] + c) % M;
                if (parent.find(nxt2) == parent.end()) {
                    parent[nxt2] = {curr, {2, c}};
                    q.push(nxt2);
                }
            }
            
            if (!found) { cout << -1 << "\n"; return 0; }
            
            vector<pair<int, int>> path;
            auto curr = found_st;
            while (parent[curr].second.first != 0) {
                path.push_back(parent[curr].second);
                curr = parent[curr].first;
            }
            reverse(path.begin(), path.end());
            for (auto op : path) {
                if (op.first == 1) for (int i = 0; i < M; i++) st[i] ^= op.second;
                else for (int i = 0; i < M; i++) st[i] = (st[i] + op.second) % M;
                apply_op(op.first, op.second);
            }
        }
    }

    int D = a ^ b;
    vector<int> parent_v(N, -1), path_c(N, -1), depth(N, 1e9);
    queue<int> Q_v;
    Q_v.push(D);
    parent_v[D] = D; depth[D] = 0;

    while (!Q_v.empty()) {
        int v = Q_v.front(); Q_v.pop();
        for (int c = 1; c < N; c++) {
            int u = (v ^ c) - c;
            if (u < 0) u = (u % N + N) % N;
            if (parent_v[u] == -1) {
                parent_v[u] = v; path_c[u] = c; depth[u] = depth[v] + 1;
                Q_v.push(u);
            }
        }
    }

    auto do_direct_swap = [&](int x, int y) {
        int X = x ^ y;
        vector<int> path;
        int cur = X;
        while (cur != D) { path.push_back(path_c[cur]); cur = parent_v[cur]; }
        
        apply_op(1, x);
        for (int c : path) { apply_op(2, c); apply_op(1, c); }
        apply_op(1, a);
        
        apply_op(0, 0);
        
        apply_op(1, a);
        for (int i = (int)path.size() - 1; i >= 0; i--) {
            int c = path[i];
            apply_op(1, c); apply_op(2, (N - c) & N_mask);
        }
        apply_op(1, x);
    };

    for (int i = 0; i < N; i++) {
        while (A[i] != i) {
            int u = A[i], v = A[u];
            int tz = __builtin_ctz(u ^ v);
            if (tz == K) {
                do_direct_swap(u, v);
            } else {
                int best_w = -1, min_d = 1e9;
                for (int w = 0; w < N; w++) {
                    if (__builtin_ctz(u ^ w) == K && __builtin_ctz(w ^ v) == K) {
                        if (parent_v[u ^ w] != -1 && parent_v[w ^ v] != -1) {
                            int d = depth[u ^ w] + depth[w ^ v];
                            if (d < min_d) { min_d = d; best_w = w; }
                        }
                    }
                }
                if (best_w != -1) {
                    do_direct_swap(u, best_w);
                    do_direct_swap(best_w, v);
                    do_direct_swap(u, best_w);
                }
            }
        }
    }

    vector<pair<int, int>> final_ops;
    for (auto op : raw_ops) {
        if (op.first != 0 && op.second == 0) continue;
        bool merged = false;
        while (!final_ops.empty() && final_ops.back().first == op.first && op.first != 0) {
            if (op.first == 1) op.second ^= final_ops.back().second;
            else if (op.first == 2) op.second = (op.second + final_ops.back().second) & N_mask;
            final_ops.pop_back();
            if (op.second == 0) { merged = true; break; }
        }
        if (!merged && (op.first == 0 || op.second != 0)) final_ops.push_back(op);
    }

    cout << final_ops.size() << "\n";
    for (auto op : final_ops) {
        if (op.first == 0) cout << 0 << "\n";
        else cout << op.first << " " << op.second << "\n";
    }

    return 0;
}


## B 长跑

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

struct Station {
    int p;
    int c;
    
    bool operator<(const Station& other) const {
        return p < other.p;
    }
};

void solve() {
    int N, L, Maxn, S;
    while (cin >> N >> L >> Maxn >> S) {
        vector<Station> stations(N);
        for (int i = 0; i < N; ++i) {
            cin >> stations[i].p >> stations[i].c;
        }
        
        sort(stations.begin(), stations.end());
        
        vector<int> dp(S + 1, -1);
        dp[0] = Maxn;
        
        for (int i = 0; i < N; ++i) {
            int p = stations[i].p;
            int cost = stations[i].c;
            
            for (int c = S; c >= cost; --c) {
                if (dp[c - cost] >= p) {
                    dp[c] = max(dp[c], p + Maxn);
                }
            }
        }
        
        bool possible = false;
        for (int c = 0; c <= S; ++c) {
            if (dp[c] >= L) {
                possible = true;
                break;
            }
        }
        
        if (possible) {
            cout << "Yes\n";
        } else {
            cout << "No\n";
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    solve();
    
    return 0;
}


## C 最长回文

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

const int P1 = 131, M1 = 1e9 + 7;
const int P2 = 13331, M2 = 1e9 + 9;

vector<long long> p1_pow, p2_pow;

void precompute_powers(int n) {
    p1_pow.assign(n + 1, 1);
    p2_pow.assign(n + 1, 1);
    for (int i = 1; i <= n; ++i) {
        p1_pow[i] = (p1_pow[i - 1] * P1) % M1;
        p2_pow[i] = (p2_pow[i - 1] * P2) % M2;
    }
}

struct HashArray {
    vector<long long> h1, h2;
    HashArray(const string& s) {
        int n = s.length();
        h1.assign(n + 1, 0);
        h2.assign(n + 1, 0);
        for (int i = 1; i <= n; ++i) {
            h1[i] = (h1[i - 1] * P1 + s[i - 1]) % M1;
            h2[i] = (h2[i - 1] * P2 + s[i - 1]) % M2;
        }
    }
    pair<long long, long long> get(int l, int r) {
        if (l > r) return {0, 0};
        long long v1 = (h1[r] - h1[l - 1] * p1_pow[r - l + 1]) % M1;
        if (v1 < 0) v1 += M1;
        long long v2 = (h2[r] - h2[l - 1] * p2_pow[r - l + 1]) % M2;
        if (v2 < 0) v2 += M2;
        return {v1, v2};
    }
};

vector<int> manacher(const string& s) {
    string T = "^#";
    for (char c : s) {
        T += c;
        T += '#';
    }
    T += '$';
    int m = T.length();
    vector<int> P(m, 0);
    int C = 0, R = 0;
    for (int i = 2; i < m - 2; ++i) {
        int i_mirror = 2 * C - i;
        if (R > i) P[i] = min(R - i, P[i_mirror]);
        while (T[i + 1 + P[i]] == T[i - 1 - P[i]]) P[i]++;
        if (i + P[i] > R) {
            C = i;
            R = i + P[i];
        }
    }
    return P;
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    int n;
    if (!(cin >> n)) return 0;
    string A, B;
    cin >> A >> B;

    precompute_powers(n);

    string A_rev = A;
    reverse(A_rev.begin(), A_rev.end());

    HashArray HA(A_rev);
    HashArray HB(B);

    auto get_lcp = [&](int idx_A, int idx_B) {
        if (idx_A == 0 || idx_B > n) return 0;
        int low = 1, high = min(idx_A, n - idx_B + 1);
        int ans = 0;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            int lA = n - idx_A + 1;
            int rA = lA + mid - 1;
            int lB = idx_B;
            int rB = lB + mid - 1;
            
            if (HA.get(lA, rA) == HB.get(lB, rB)) {
                ans = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        return ans;
    };

    vector<int> PA = manacher(A);
    vector<int> PB = manacher(B);

    int max_len = 0;

    for (int i = 2; i <= 2 * n; ++i) {
        if (PA[i] == 0) continue;
        int k = (i + PA[i] - 1) / 2;
        int l = (i - PA[i]) / 2 + 1;
        int match = get_lcp(l - 1, k); 
        max_len = max(max_len, PA[i] + 2 * match);
    }

    for (int i = 2; i <= 2 * n; ++i) {
        if (PB[i] == 0) continue;
        int r = (i + PB[i] - 1) / 2;
        int k = (i - PB[i]) / 2 + 1;
        int match = get_lcp(k, r + 1);
        max_len = max(max_len, PB[i] + 2 * match);
    }

    for (int k = 1; k <= n; ++k) {
        int match = get_lcp(k, k);
        max_len = max(max_len, 2 * match);
    }

    cout << max_len << "\n";
    return 0;
}


## D 优惠券

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <set>
#include <unordered_map>
#include <vector>

using namespace std;

void solve() {
    int m;
    
    unordered_map<int, bool> held;   
    unordered_map<int, int> last_op; 
    vector<int> modified_x;          

    while (cin >> m) {
        int error_line = -1;
        set<int> available_q; 

        for (int i = 1; i <= m; ++i) {
            string op;
            cin >> op;
            
            int x = -1;
            if (op == "I" || op == "i" || op == "O" || op == "o") {
                cin >> x;
                if (last_op[x] == 0) {
                    modified_x.push_back(x);
                }
            }

            if (error_line != -1) continue;

            if (op == "?" || op == "？") {
                available_q.insert(i);
            } 
            else if (op == "I" || op == "i") {
                if (held[x]) {
                    auto it = available_q.upper_bound(last_op[x]);
                    if (it != available_q.end()) {
                        available_q.erase(it);
                        last_op[x] = i; 
                    } else {
                        error_line = i;
                    }
                } else {
                    held[x] = true;
                    last_op[x] = i;
                }
            } 
            else if (op == "O" || op == "o") {
                if (!held[x]) {
                    auto it = available_q.upper_bound(last_op[x]);
                    if (it != available_q.end()) {
                        available_q.erase(it);
                        last_op[x] = i;
                    } else {
                        error_line = i;
                    }
                } else {
                    held[x] = false;
                    last_op[x] = i;
                }
            }
        }

        cout << error_line << "\n";

        for (int k : modified_x) {
            held[k] = false;
            last_op[k] = 0;
        }
        modified_x.clear();
        available_q.clear();
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    solve();
    
    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <iostream>
#include <vector>

using namespace std;

struct DSU {
    vector<int> parent;
    DSU(int n) {
        parent.resize(n);
        for (int i = 0; i < n; ++i) {
            parent[i] = i;
        }
    }
    
    int find(int i) {
        if (parent[i] == i) return i;
        return parent[i] = find(parent[i]);
    }
        void unite(int i, int j) {
        int root_i = find(i);
        int root_j = find(j);
        if (root_i != root_j) {
            parent[root_i] = root_j;
        }
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    int n;
    if (!(cin >> n)) return 0;
    
    vector<pair<int, int>> points(n);
    for (int i = 0; i < n; ++i) {
        cin >> points[i].first >> points[i].second;
    }
    
    DSU dsu(n);
    
    for (int i = 0; i < n; ++i) {
        for (int j = i + 1; j < n; ++j) {
            if (points[i].first == points[j].first || points[i].second == points[j].second) {
                dsu.unite(i, j);
            }
        }
    }
    
    int components = 0;
    for (int i = 0; i < n; ++i) {
        if (dsu.parent[i] == i) {
            components++;
        }
    }
    
    cout << components - 1 << "\n";
    
    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <vector>

using namespace std;

const long long MOD = 1000000007; 
const long long B = 131;          

vector<long long> p_pow;

void precompute() {
    p_pow.resize(200005);
    p_pow[0] = 1;
    for (int i = 1; i < 200005; ++i) {
        p_pow[i] = (p_pow[i - 1] * B) % MOD;
    }
}

bool match_exact(const string& S, int start, const string& C) {
    for (int i = 0; i < (int)C.length(); ++i) {
        if (C[i] != '?' && C[i] != S[start + i]) return false;
    }
    return true;
}

int find_chunk(const string& S, const vector<long long>& h_S, int ptr, int max_start, const string& C) {
    if (C.length() == 0) return ptr; 
    
    long long h_C = 0;
    vector<int> q_pos; 
    
    for (int i = 0; i < (int)C.length(); ++i) {
        if (C[i] == '?') {
            q_pos.push_back(i);
            h_C = (h_C * B) % MOD; 
        } else {
            h_C = (h_C * B + C[i]) % MOD;
        }
    }
    
    int len = C.length();
    for (int j = ptr; j <= max_start; ++j) {
        long long cand_h = (h_S[j + len] - (h_S[j] * p_pow[len]) % MOD) % MOD;
        if (cand_h < 0) cand_h += MOD;

        for (int q : q_pos) {
            long long char_val = S[j + q];
            long long subtract = (char_val * p_pow[len - 1 - q]) % MOD;
            cand_h = (cand_h - subtract) % MOD;
            if (cand_h < 0) cand_h += MOD;
        }

        if (cand_h == h_C) {
            bool ok = true;
            for (int k = 0; k < len; ++k) {
                if (C[k] != '?' && C[k] != S[j + k]) {
                    ok = false;
                    break;
                }
            }
            if (ok) return j; 
        }
    }
    return -1;
}

void solve() {
    string P;
    if (!(cin >> P)) return;
    int n;
    cin >> n;

    precompute();

    vector<string> chunks;
    string curr = "";
    for (char c : P) {
        if (c == '*') {
            chunks.push_back(curr);
            curr = "";
        } else {
            curr += c;
        }
    }
    chunks.push_back(curr);

    int k = chunks.size() - 1; 
    long long sum_len = 0;     
    for (const string& C : chunks) {
        sum_len += C.length();
    }

    for (int i = 0; i < n; ++i) {
        string S;
        cin >> S;

        if (k == 0) {
            if (S.length() == chunks[0].length() && match_exact(S, 0, chunks[0])) {
                cout << "YES\n";
            } else {
                cout << "NO\n";
            }
            continue;
        }

        if (S.length() < sum_len) {
            cout << "NO\n";
            continue;
        }

        if (!match_exact(S, 0, chunks[0])) {
            cout << "NO\n";
            continue;
        }

        int suffix_start = S.length() - chunks.back().length();
        if (!match_exact(S, suffix_start, chunks.back())) {
            cout << "NO\n";
            continue;
        }

        vector<long long> h_S(S.length() + 1, 0);
        for (int j = 0; j < (int)S.length(); ++j) {
            h_S[j + 1] = (h_S[j] * B + S[j]) % MOD;
        }

        int ptr = chunks[0].length();
        int max_end = suffix_start; 
        bool possible = true;

        for (int j = 1; j < k; ++j) {
            const string& C_j = chunks[j];
            int max_start = max_end - C_j.length();
            
            int match_pos = find_chunk(S, h_S, ptr, max_start, C_j);
            
            if (match_pos == -1) {
                possible = false;
                break;
            }
            ptr = match_pos + C_j.length();
        }

        if (possible) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    solve();
    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <vector>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    int prio[3][3] = {0};
    for (int i = 0; i < 6; ++i) {
        string s;
        cin >> s;
        int u = s[0] - 'A';
        int v = s[1] - 'A';
        prio[u][v] = i; 
    }

    int to[35][3];
    long long cnt[35][3];

    for (int p = 0; p < 3; ++p) {
        int p1 = (p + 1) % 3;
        int p2 = (p + 2) % 3;
        if (prio[p][p1] < prio[p][p2]) {
            to[1][p] = p1;
        } else {
            to[1][p] = p2;
        }
        cnt[1][p] = 1;
    }

    for (int k = 2; k <= n; ++k) {
        for (int p = 0; p < 3; ++p) {
            int X = p;                 
            int Y = to[k - 1][X];      
            int Z = 3 - X - Y;         
            int W = to[k - 1][Y];     

            if (W == Z) {
                to[k][X] = Z;
                cnt[k][X] = cnt[k - 1][X] + 1 + cnt[k - 1][Y];
            } else if (W == X) {
                to[k][X] = Y;
                cnt[k][X] = cnt[k - 1][X] + 1 + cnt[k - 1][Y] + 1 + cnt[k - 1][X];
            }
        }
    }

    cout << cnt[n][0] << "\n";

    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <iostream>
#include <cmath>
#include <algorithm>

using namespace std;

void solve() {
    long long xp, yp, xs, ys;
    if (!(cin >> xp >> yp >> xs >> ys)) return;

    long long dx = abs(xs - xp);
    long long dy = abs(ys - yp);
    long long x = max(dx, dy);
    long long y = min(dx, dy);

    if (x == 0 && y == 0) {
        cout << 0 << "\n";
        return;
    }
    if (x == 1 && y == 0) {
        cout << 3 << "\n";
        return;
    }
    if (x == 2 && y == 2) {
        cout << 4 << "\n";
        return;
    }

    // ceil(a/b) 可以写成 (a+b-1)/b
    long long ans = max((x + 1) / 2, (x + y + 2) / 3);

    if ((ans % 2) != ((x + y) % 2)) {
        ans++;
    }

    cout << ans << "\n";
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    solve();
    
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
#include <vector>
#include <stack>
#include <algorithm>

using namespace std;

class Solution {
public:
    /**
     *
     * * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();
        if (n == 0) return 0;

        vector<int> h(n + 2, 0);
        for (int i = 0; i < n; ++i) {
            h[i + 1] = heights[i];
        }

        int max_area = 0;

        for (int i = 0; i < h.size(); ++i) {
            while (!st.empty() && h[i] < h[st.top()]) {
                int current_height = h[st.top()];
                st.pop();
                
                int left_bound = st.top();
                int right_bound = i;
                
                int current_width = right_bound - left_bound - 1;
                max_area = max(max_area, current_height * current_width);
            }
            st.push(i);
        }

        return max_area;
    }
};

## J 消防局的设立

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    vector<int> p(n + 1, 0);
    for (int i = 2; i <= n; ++i) {
        cin >> p[i];
    }

    vector<int> req(n + 1, 0);     
    vector<int> cov(n + 1, 1e9);   
    int ans = 0;

    for (int i = n; i >= 1; --i) {
        if (req[i] + cov[i] <= 2) {
            req[i] = -1; // 抹除覆盖需求
        }
        
        if (req[i] == 2) {
            ans++;
            cov[i] = 0;  // 当前节点有了消防局
            req[i] = -1; // 所有需求被满足
        }

        if (i > 1) {
            int parent = p[i];
            if (req[i] != -1) {
                req[parent] = max(req[parent], req[i] + 1);
            }
            cov[parent] = min(cov[parent], cov[i] + 1);
        }
    }

    if (req[1] >= 0) {
        ans++;
    }

    cout << ans << "\n";

    return 0;
}